In [2]:
let ov : Vec<Option<f64>> = vec![Some(1.0), Some(2.0), None, Some(4.0), Some(5.0)];
ov

[Some(1.0), Some(2.0), None, Some(4.0), Some(5.0)]

In [3]:
ov.iter().map(|opt| opt.map(|x| x + 1.0)).collect::<Vec<_>>()

[Some(2.0), Some(3.0), None, Some(5.0), Some(6.0)]

In [4]:
let fv : Vec<f64> = vec![1.0, 2.0, f64::NAN, 4.0, 5.0];
fv

[1.0, 2.0, NaN, 4.0, 5.0]

In [5]:
fv.iter().map (
    |x| if f64::is_nan(*x) { None } else { Some(*x) }
).collect::<Vec<_>>()


[Some(1.0), Some(2.0), None, Some(4.0), Some(5.0)]

In [6]:
// Streaming EMA filter — single Option-centric step, mirroring calc_ema:
// a None input resets the run (gap), output is None during warmup, then Some.
pub struct ExponentialMovingAverage {
    period: i64,
    alpha: f64,
    ema: f64,
    count: i64,
}

impl ExponentialMovingAverage {
    pub fn new(period: i64) -> Result<Self, String> {
        if period <= 0 {
            return Err("period must be > 0".to_string());
        }
        Ok(Self {
            period,
            alpha: 2.0 / (period as f64 + 1.0),
            ema: f64::NAN,
            count: 0,
        })
    }

    pub fn next(&mut self, input: Option<f64>) -> Option<f64> {
        let Some(value) = input else {
            // Null breaks the current run.
            self.ema = f64::NAN;
            self.count = 0;
            return None;
        };

        if self.count == 0 {
            self.ema = value;
        } else {
            self.ema += self.alpha * (value - self.ema);
        }
        self.count += 1;

        (self.count >= self.period).then_some(self.ema)
    }
}

In [7]:
// Plain f64 input: wrap each value in Some(..). period=3 -> first two are warmup (None).
// (explicit type: evcxr persists top-level lets and needs the concrete type)
let fv: Vec<f64> = vec![1.0, 2.0, 3.0, 4.0, 5.0];

let mut ema: ExponentialMovingAverage = ExponentialMovingAverage::new(3).unwrap();

fv.iter().map(|x| ema.next(Some(*x))).collect::<Vec<_>>()

[None, None, Some(2.25), Some(3.125), Some(4.0625)]

In [8]:
// Option input flows straight in. period=2 to show warmup + the None reset.
let ov: Vec<Option<f64>> = vec![Some(1.0), Some(2.0), None, Some(4.0), Some(5.0)];

let mut ema: ExponentialMovingAverage = ExponentialMovingAverage::new(2).unwrap();

ov.iter().map(|x| ema.next(*x)).collect::<Vec<_>>()

[None, Some(1.6666666666666665), None, None, Some(4.666666666666667)]